In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

In [ ]:
vmin, vmax = -10., 10.
n_sup = 51
support = np.linspace(vmin, vmax, n_sup)

# Uniform distribution
probs = np.ones(n_sup)
probs /= probs.sum()
z = torch.from_numpy(probs).float()
plt.bar(support, probs)

In [ ]:
support

In [ ]:
probs

In [ ]:
def project_val(val, N, clip=(-10, 10)):
    vmin, vmax = clip[0], clip[1]
    dz = (vmax - vmin) / (N - 1)
    bj = (val - vmin) / dz
    bj = np.round(bj)
    bj = int(np.clip(bj, 0, N - 1))
    return bj

def update_dist(r, probs, lim=(-10, 10), gamma=0.8):
    n_sup = probs.shape[0]
    bj = project_val(r, n_sup, lim) #value of observed reward in the support
    m = probs.clone()
    j = 1

    # starting from immediate left neighbor
    for i in range(bj, 1, -1):
        m[i] += np.power(gamma, j) * m[i-1]
        j += 1
    
    j = 1
    # starting from immediate right position
    for i in range(bj, n_sup -1, 1):
        m[i] += np.power(gamma, j) * m[i+1]
        j += 1

    m /= m.sum()
    return m
    




In [ ]:
obs_reward = -1
z = update_dist(obs_reward, z, lim=(vmin, vmax), gamma=0.1)
z

In [ ]:
plt.bar(support, z)

In [ ]:
obs_rewards = [10, 10, 10, 10, 10, 10, 10, 10, 10]

for i in range(len(obs_rewards)):
    z = update_dist(obs_reward, z, lim=(vmin, vmax), gamma=0.7)
plt.bar(support, z)

In [ ]:
from collections import deque

state_size = 128
action_size = 3 #UP, DOWN, NO-OP
vmin, vmax = -10, 10
n_sup = 51
gamma = 0.9
lr = 1e-4
update_rate = 75
display_rate = 10

# replay buffer
replay_size = 200

# epsilon-greedy
epsilon = 1.0
epsilon_min = 0.1


batch_size = 10

support = torch.linspace(vmin, vmax, n_sup)

### Manual Parameter Tracking

In [ ]:
h_dims = [100, 25]
dims = [state_size] + h_dims + [n_sup]

# The last dimension outputs n_sup=51 probability values for each action
# The last layer is multiplied by action_size = 3 seperate matrices
# to get action_size seperate distributions
# we therefore need to account for that in the weights
total_params = sum(
    dims[i] * dims[i + 1] * (action_size if i == len(dims) - 2 else 1)
    for i in range(len(dims) - 1)
)

In [ ]:
#B : batch_size
#A : action_size
#S : state_size
#N : n_sup

class Network:
    def __init__(self, state_size, action_size, dims, total_params, device):
        self.state_size = state_size
        self.action_size = action_size
        self.dims = dims
        self.device = device
        self.total_params = total_params
        self.parameters = (torch.randn(total_params) / 10).to(device)

    def forward(self, x):

        # x : (B, S)
        # expected output: (B, A, N)

        single = (x.dim() == 1)
        if single:
            x = x.unsqueeze(0)

        x = x.to(self.device).float() / 255.0 

        offset = 0

        # Shared layers
        for i in range(len(self.dims) - 2):

            n_params = self.dims[i] * self.dims[i + 1]

            weights = self.parameters[
                offset:offset + n_params
            ].reshape(self.dims[i], self.dims[i + 1])

            x = x @ weights #(B, self.dims[i+1])
            x = F.selu(x)

            offset += n_params

        # Separate output matrix for each action
        out = []

        src_dim = self.dims[-2]
        out_dim = self.dims[-1]

        step = src_dim * out_dim

        for i in range(self.action_size):

            weights = self.parameters[
                offset + i * step:
                offset + (i + 1) * step
            ].reshape(src_dim, out_dim)

            out_i = x @ weights 

            out.append(out_i)

        out = torch.stack(out, dim=1) #(B, A, N)

        out = F.softmax(out, dim=2) #(B, A, N)

        return out.squeeze(0) if single else out

    def __call__(self, x):
        return self.forward(x)

    def __len__(self):
        return self.total_params

In [ ]:
import random

In [ ]:
def lossfn(x, y):
    # x, y: (B, N) predicted and target distributions
    return -(y * torch.log(x.clamp_min(1e-8))).sum(dim=1).mean()



In [ ]:
#B : batch_size
#A : action_size
#S : state_size
#N : n_sup

class FreewayAgent:
    def __init__(self, net_dict, device, lr=1e-4, buffer_size=200, vmin=-10, vmax=10, n_sup=51):
        self.state_size = net_dict['state_size']
        self.action_size = net_dict['action_size']
        self.device = device
        self.lr = lr
        self.support = torch.linspace(vmin, vmax, n_sup).to(device)
        self.n_sup = n_sup
        self.vmin, self.vmax = vmin, vmax
        self.dz = (vmax - vmin) / (self.n_sup - 1)
        self.main_network = Network(**net_dict)
        self.target_network = Network(**net_dict)
        self.main_network.parameters = self.main_network.parameters.to(device)
        self.target_network.parameters = self.target_network.parameters.to(device)

        self.target_network.parameters = self.main_network.parameters.clone()

        self.main_network.parameters.requires_grad_(True)
        self.target_network.parameters.requires_grad_(False)

        self.memory = deque(maxlen=buffer_size)

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def select_action(self, state, epsilon):
        if np.random.uniform(0, 1) < epsilon:
            action = np.random.randint(0, self.action_size)
        else:
            out = self.main_network(state)
            q_values = out @ self.support # (self.action_size,)
            action = torch.argmax(q_values).item()
        return action

    def _get_target_distribution(self, batch_size, gamma, rewards, dones, next_states):
        with torch.no_grad():
            # next_states: (B, S)
            next_probs = self.target_network(next_states) #(B x A x N)
            next_q_values = next_probs @ self.support #(B x A)
            best_actions = torch.argmax(next_q_values, dim=1) #(B,)

            batch_idx = torch.arange(batch_size, device=self.device) #(B,)
            next_dists = next_probs[batch_idx, best_actions] #(B,N)

            # Bellman tranformation
            # rewards: (B,), dones: (B,)
            # self.support: (N,)
            Tz = rewards.unsqueeze(1) + gamma * (1 - dones).unsqueeze(1) * self.support.unsqueeze(0)
            Tz = Tz.clamp(self.vmin, self.vmax) #(B, N)

            b = (Tz - self.vmin) / self.dz #(B, N)

            lo = b.floor().long() #(B,N)
            hi = b.ceil().long() #(B,N)

            # fix disappearing / double-counted probability mass when b lands
            # exactly on an integer (lo == hi). Shift lo down when possible;
            # only shift hi up when lo is already at the floor (0). Never
            # shift both, or mass gets double-counted into two bins.
            eq_mask = (lo == hi)
            shift_lo = eq_mask & (lo > 0)
            shift_hi = eq_mask & (lo == 0)

            lo[shift_lo] -= 1
            hi[shift_hi] += 1

            lo = lo.clamp(0, self.n_sup - 1)
            hi = hi.clamp(0, self.n_sup - 1)

            m = torch.zeros(batch_size, self.n_sup, device=self.device) #(B, N)

            for i in range(batch_size):
                for j in range(self.n_sup):
                    p = next_dists[i, j]
                    m[i, lo[i, j]] += p * (hi[i, j] - b[i, j])
                    m[i, hi[i, j]] += p * (b[i, j] - lo[i, j])

            assert torch.isclose(m[0].sum(), torch.tensor(1.0, device=self.device), atol=1e-4), \
                    f"expected probability distribution \
                    per row for all {m.shape[0]} rows. got {m[0].sum():.2f} for first row sum."
            return m

    def train(self, gamma, batch_size):
        if len(self.memory) < batch_size:
            return None
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.stack(states).to(self.device) #(B, S)
        next_states = torch.stack(next_states).to(self.device) #(B, S)
        actions = torch.tensor(actions, dtype=torch.long, device=self.device).unsqueeze(1) #(B, 1)
        rewards = torch.tensor(rewards, dtype=torch.float32, device=self.device) #(B,)
        dones = torch.tensor(dones, dtype=torch.float32, device=self.device) #(B,)

        batch_idx = torch.arange(batch_size, device=self.device) #(B,)

        pred_dists = self.main_network(states) #(B, A, N)
        pred_dists = pred_dists[batch_idx, actions.squeeze(1)] #(B, N)

        with torch.no_grad():
            target_dists = self._get_target_distribution(batch_size, gamma, rewards, dones, next_states)

        loss = lossfn(pred_dists, target_dists)

        if self.main_network.parameters.grad is not None:
            self.main_network.parameters.grad.zero_()
        loss.backward()

        with torch.no_grad():
            self.main_network.parameters -= self.lr * self.main_network.parameters.grad

        return loss.item()

    def update_target_network(self, num_iter, update_rate):
        if num_iter % update_rate == 0:
            with torch.no_grad():
                self.target_network.parameters = self.main_network.parameters.clone()
            self.target_network.parameters.requires_grad_(False)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

net_dict = {
    'state_size': state_size,
    'action_size': action_size,
    'dims': dims,
    'total_params': total_params,
    'device': device
}

agent = FreewayAgent(net_dict, device, lr=lr, buffer_size=replay_size)

### Training Loop

In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

In [ ]:
state_size = 128
action_size = 3  # UP, DOWN, NO-OP
vmin, vmax = -10, 10
n_sup = 51
gamma = 0.9
lr = 1e-4
update_rate = 75
display_rate = 10

# epsilon-greedy (exponential decay)
epsilon_start = 1.0
epsilon_min = 0.1
epsilon_decay = 0.0005   

batch_size = 10
losses = []
stats_rewards_list = []
epsilon = epsilon_start
num_iter = 0
total_steps = 0
num_episodes = 1300
num_timesteps = 200

env = gym.make('ALE/Freeway-v5', obs_type="ram")
stats_every = 100

for ep in range(num_episodes):
    state, info = env.reset()
    episode_loss = []
    total_reward = 0

    for t in range(num_timesteps):
        state_tensor = torch.from_numpy(state)
        action = agent.select_action(state_tensor, epsilon)
        next_state, reward, done, truncated, info = env.step(action)
        total_reward += reward
        total_steps += 1

        agent.memory.append((
            state_tensor,
            action,
            reward,
            torch.from_numpy(next_state),
            done
        ))

        state = next_state

        loss = agent.train(gamma, batch_size)

        
        epsilon = epsilon_min + (epsilon_start - epsilon_min) * np.exp(-epsilon_decay * total_steps)

        agent.update_target_network(total_steps, update_rate)

        if loss is not None:
            episode_loss.append(loss)
            losses.append(loss)

        num_iter += 1
        if num_iter % 10 == 0:
            print(
                f"Episode {ep:4d} | "
                f"Step {t:3d} | "
                f"Iter {num_iter:6d} | "
                f"Loss: {loss:.4f} | "
                f"Epsilon: {epsilon:.3f}"
            )

        if done or truncated:
            break

    stats_rewards_list.append((ep, total_reward, t))

    if ep % stats_every == 0:
        avg_reward = np.mean([r[1] for r in stats_rewards_list[-stats_every:]])
        avg_loss = np.mean(episode_loss) if episode_loss else 0.0
        print(f"Episode {ep} | steps {total_steps} | reward {total_reward:.1f} "
              f"| avg_reward(last {stats_every}) {avg_reward:.1f} "
              f"| epsilon {epsilon:.3f} | loss {avg_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def moving_average(data, window=50):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

#Loss (raw, per training step)
axes[0, 0].plot(losses, alpha=0.3, label='raw loss')
if len(losses) > 50:
    smoothed = moving_average(losses, window=50)
    axes[0, 0].plot(range(len(losses) - len(smoothed), len(losses)), smoothed, label='moving avg (50)')
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Training step')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

#Reward per episode
episodes = [r[0] for r in stats_rewards_list]
rewards = [r[1] for r in stats_rewards_list]
axes[0, 1].plot(episodes, rewards, alpha=0.3, label='raw reward')
if len(rewards) > 20:
    smoothed_r = moving_average(rewards, window=20)
    axes[0, 1].plot(episodes[-len(smoothed_r):], smoothed_r, label='moving avg (20)')
axes[0, 1].set_title('Reward per Episode')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Total reward')
axes[0, 1].legend()

#Episode length (steps survived)
ep_lengths = [r[2] for r in stats_rewards_list]
axes[1, 0].plot(episodes, ep_lengths, alpha=0.5)
axes[1, 0].set_title('Episode Length (steps)')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Steps survived')

# Epsilon decay over episodes (sanity check)
# reconstruct epsilon trajectory from total_steps if you didn't log it directly
# if you have an epsilon history list, plot that instead
axes[1, 1].plot(episodes, rewards)
axes[1, 1].axhline(y=np.mean(rewards[-50:]) if len(rewards) >= 50 else np.mean(rewards),
                     color='r', linestyle='--', label='recent avg')
axes[1, 1].set_title('Reward (recent average highlighted)')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Total reward')
axes[1, 1].legend()

plt.tight_layout()
plt.show()